# Projet IC2 (3/3) — Intervalle conforme, SHAP, rapport

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/inphb/ic2/notebooks/IC2_Projet_S5_incertitude_shap.ipynb)
> Données : dossier `IC2_TP_donnees`. Bibliothèques : pandas, numpy, matplotlib, scikit-learn, torch, shap.

In [ ]:
# Google Colab : installe SHAP si besoin (quelques secondes).
import sys
if "google.colab" in sys.modules:
    %pip install -q shap

In [ ]:
# Configuration : Colab ou Jupyter local. Exécutez cette cellule en premier.
import os, shutil, urllib.request
DONNEES = ['denoyage_fosse.csv']
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/inphb/ic2/data/'
for f in DONNEES:
    if os.path.exists(f):
        continue
    if os.path.exists(os.path.join('..', 'data', f)):
        shutil.copy(os.path.join('..', 'data', f), f)   # kit hors ligne / dépôt cloné
    else:
        urllib.request.urlretrieve(URL + f, f)            # Colab : téléchargement
    print('OK', f)


In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import train_test_split, KFold, GroupKFold
from sklearn.metrics import r2_score, mean_squared_error
import shap
pit = pd.read_csv('denoyage_fosse.csv', parse_dates=['semaine'])
f = pd.DataFrame(index=pit.index)
for c in ['P1_fosse', 'P2_150m', 'P3_600m']:
    for lag in (0, 1, 2, 4): f[f'{c}_l{lag}'] = pit[c].shift(lag)
f['pluie_4s'] = pit['pluie_mm'].rolling(4).sum(); f['pluie_12s'] = pit['pluie_mm'].rolling(12).sum()
f['pompage_t'] = pit['pompage_m3h']; f['pompage_plan_t4'] = pit['pompage_m3h'].shift(-4)
j = pit['semaine'].dt.dayofyear; f['sin_saison'] = np.sin(2*np.pi*j/365); f['cos_saison'] = np.cos(2*np.pi*j/365)
f['cible'] = pit['P1_fosse'].shift(-4); f['delta'] = f['cible'] - pit['P1_fosse']
f = f.dropna()
date = pit.loc[f.index, 'semaine']; tr, te = f[date < '2024-01-01'], f[date >= '2024-01-01']
X = [c for c in f.columns if c not in ('cible', 'delta')]
rmse = lambda a, b: np.sqrt(np.mean((np.asarray(a) - np.asarray(b))**2))
print(len(tr), 'semaines d’entraînement,', len(te), 'de test')

## 1. Intervalle conforme à 90 % (calibration = dernière année d’entraînement)

In [ ]:
cal = tr.index[int(len(tr) * 0.8):]; app = tr.index[:int(len(tr) * 0.8)]
g = HistGradientBoostingRegressor(max_iter=500, learning_rate=0.05, max_leaf_nodes=15, random_state=0).fit(tr.loc[app, X], tr.loc[app, 'delta'])
q = np.quantile(np.abs(tr.loc[cal, 'delta'] - g.predict(tr.loc[cal, X])), 0.9)
prev = te['P1_fosse_l0'] + g.predict(te[X])
couv = np.mean(np.abs(te['cible'] - prev) <= q)
print('demi-largeur :', round(q, 2), 'm ; couverture test :', round(couv, 2))
plt.fill_between(range(len(te)), prev - q, prev + q, alpha=.3); plt.plot(prev.values); plt.plot(te['cible'].values, 'k'); plt.show()

**Question :** la couverture est inférieure à 90 %. Quelle hypothèse est violée ? Que proposez-vous en exploitation ?

## 2. SHAP

In [ ]:
gb = HistGradientBoostingRegressor(max_iter=500, learning_rate=0.05, max_leaf_nodes=15, random_state=0).fit(tr[X], tr['delta'])
sv = shap.TreeExplainer(gb).shap_values(te[X])
shap.summary_plot(sv, te[X], max_display=10)

## 3. Rapport
Structure IC1 + sections : protocole temporel, extrapolation, intervalle et couverture, cohérence physique (SHAP).